In [1]:
import os
METADATA_PATH = os.path.join("..", "data_bci2a_processed_1000", "dataset_metadata_bci2a.csv")
OUTPUT_DIR = os.path.join("..", "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [3]:
"""
BCI Competition IV-2a Strict Outer Leave-One-Subject-Out (LOSO) Training & Evaluation Script
Compatible with Python 3.10 & TensorFlow 2.10.1
Author: Armin Abdollahi
"""

import os
import sys

# Crucial Windows CuDNN fix for TF 2.10.1
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_CUBLAS_TENSOR_OP_MATH_FP32"] = "0"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

import json
import time
import random
import argparse
import gc
import numpy as np
import pandas as pd
import tensorflow as tf

# Configure GPU Memory Growth dynamically
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(f"GPU config error: {e}")

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import (
    Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
    Dropout, GlobalAveragePooling1D, Conv1D, TimeDistributed,
    Conv2D, DepthwiseConv2D, SeparableConv2D, Flatten, AveragePooling2D
)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


class BCI2aDataGenerator(Sequence):
    def __init__(self, df, model_type, batch_size=32, shuffle=True, normalization="none"):
        self.df = df.reset_index(drop=True)
        self.model_type = model_type.lower()
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.normalization = normalization
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_idx]

        X, y = [], []
        for _, row in batch_df.iterrows():
            data = np.load(row["file"])
            
            if data.shape != (22, 1000):
                raise ValueError(f"Expected shape (22, 1000) for file {row['file']}, got {data.shape}")
            
            if self.normalization == "per_trial_channel_zscore":
                mean = np.mean(data, axis=1, keepdims=True)
                std = np.std(data, axis=1, keepdims=True)
                data = (data - mean) / (std + 1e-6)

            if self.model_type == "eegnet":
                data = data[:, :, np.newaxis]
            elif self.model_type in ["cvt", "vit_pe"]:
                data = data.T
            else:
                raise ValueError(f"Unknown model type: {self.model_type}")

            X.append(data)
            y.append(row["label"])

        X = np.array(X, dtype=np.float32)
        y = tf.keras.utils.to_categorical(y, num_classes=2)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)


class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config


class SinusoidalPositionEmbedding(Layer):
    def __init__(self, sequence_length, dim, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.dim = dim
        self.pe = self.get_sinusoidal_embeddings(sequence_length, dim)

    def get_sinusoidal_embeddings(self, sequence_length, embedding_dim):
        position = np.arange(sequence_length)[:, np.newaxis]
        div_term = np.exp(np.arange(0, embedding_dim, 2) * -(np.log(10000.0) / embedding_dim))
        pe = np.zeros((sequence_length, embedding_dim))
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        return tf.constant(pe, dtype=tf.float32)

    def call(self, x):
        return x + self.pe

    def get_config(self):
        config = super().get_config()
        config.update({"sequence_length": self.sequence_length, "dim": self.dim})
        return config


class PatchEncoder(Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = Dense(units=projection_dim)
        pos_encoding_matrix = self.get_sinusoidal_position_embedding(num_patches, projection_dim)
        self.position_embedding = tf.Variable(pos_encoding_matrix, trainable=False, name="position_embedding")

    def get_sinusoidal_position_embedding(self, sequence_length, embedding_dim):
        angle_rates = 1.0 / (10000 ** (tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)))
        positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
        angle_rads = positions * angle_rates[tf.newaxis, :]
        return tf.concat([tf.math.sin(angle_rads), tf.math.cos(angle_rads)], axis=-1)

    def call(self, patch):
        batch_size = tf.shape(patch)[0]
        patch_height = patch.shape[-2]
        patch_width = patch.shape[-1]
        patch_flattened = tf.reshape(patch, [batch_size, self.num_patches, patch_height * patch_width])
        encoded = self.projection(patch_flattened)
        encoded += self.position_embedding
        return encoded

    def get_config(self):
        config = super().get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config


def build_eegnet(lr=0.000175, dropout_rate=0.416, f1=32, depth_multiplier=2):
    inputs = Input(shape=(22, 1000, 1))
    
    block1 = Conv2D(f1, (1, 64), padding="same", use_bias=False)(inputs)
    block1 = tf.keras.layers.BatchNormalization()(block1)
    block1 = DepthwiseConv2D((22, 1), depth_multiplier=depth_multiplier, use_bias=False, depthwise_constraint=max_norm(1.0))(block1)
    block1 = tf.keras.layers.BatchNormalization()(block1)
    block1 = tf.keras.layers.Activation("elu")(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = Dropout(dropout_rate)(block1)

    f2 = f1 * depth_multiplier
    block2 = SeparableConv2D(f2, (1, 16), use_bias=False, padding="same")(block1)
    block2 = tf.keras.layers.BatchNormalization()(block2)
    block2 = tf.keras.layers.Activation("elu")(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = Dropout(dropout_rate)(block2)

    flatten = Flatten()(block2)
    outputs = Dense(2, activation="softmax", kernel_constraint=max_norm(0.25))(flatten)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def build_cvt(lr=0.000160, dropout_rate=0.33870, patch_size=50, projection_dim=128, num_heads=4, transformer_layers=4, ff_dim=128):
    seq_len = 1000
    assert seq_len % patch_size == 0, f"Sequence length {seq_len} must be divisible by patch_size {patch_size}"
    num_patches = seq_len // patch_size

    inputs = Input(shape=(seq_len, 22))
    patches = Patches(patch_size)(inputs)

    cnn_encoder = tf.keras.Sequential([
        Input(shape=(patch_size, 22)),
        Conv1D(64, 10, padding='same', activation='relu'),
        Conv1D(64, 5, padding='same', activation='relu'),
        GlobalAveragePooling1D(),
        Dense(projection_dim)
    ])

    tokens = TimeDistributed(cnn_encoder)(patches)
    encoded = SinusoidalPositionEmbedding(num_patches, projection_dim)(tokens)

    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded)
        attn_out = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate)(x1, x1)
        x2 = attn_out + encoded
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        ffn = Dense(ff_dim, activation="gelu")(x3)
        ffn = Dropout(dropout_rate)(ffn)
        ffn = Dense(projection_dim)(ffn)
        encoded = ffn + x2

    representation = LayerNormalization(epsilon=1e-6)(encoded)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(dropout_rate)(representation)
    outputs = Dense(2, activation="softmax")(representation)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def build_vit_pe(lr=0.000173, dropout_rate=0.36094, patch_size=50, projection_dim=128, num_heads=8, transformer_layers=2, ff_dim=128):
    seq_len = 1000
    assert seq_len % patch_size == 0, f"Sequence length {seq_len} must be divisible by patch_size {patch_size}"
    num_patches = seq_len // patch_size

    inputs = Input(shape=(seq_len, 22))
    patches = Patches(patch_size)(inputs)
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attn_out = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate)(x1, x1)
        x2 = attn_out + encoded_patches
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = x3 + x2

    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    rep_dense = Dense(128, activation='relu')(representation)
    outputs = Dense(2, activation='softmax')(rep_dense)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def run_pipeline(metadata_path, output_dir, model_name="all", epochs=50, batch_size=32, patience=10, seed=42, normalization="none"):
    seed_everything(seed)
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, "models"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "results"), exist_ok=True)

    print(f"Loading metadata from {metadata_path}...")
    df = pd.read_csv(metadata_path)
    expected_cols = {"file", "label", "subject_id"}
    if not expected_cols.issubset(df.columns):
        raise ValueError(f"Metadata must contain columns {expected_cols}. Found: {df.columns.tolist()}")

    df = df.sort_values(by=["subject_id", "file"]).reset_index(drop=True)
    print(f"Loaded {len(df)} trials across {df['subject_id'].nunique()} subjects.")

    models_to_run = ["eegnet", "cvt", "vit_pe"] if model_name == "all" else [model_name]

    for m_name in models_to_run:
        print(f"\n=============================================")
        print(f"Starting LOSO Training for Model: {m_name.upper()}")
        print(f"=============================================")

        logo = LeaveOneGroupOut()
        groups = df["subject_id"]

        fold_results = []
        all_predictions = []

        for outer_fold, (train_val_idx, test_idx) in enumerate(logo.split(df, df["label"], groups)):
            print(f"\n--- Outer Fold {outer_fold + 1} / {df['subject_id'].nunique()} ---")
            
            train_val_df = df.iloc[train_val_idx].copy()
            test_df = df.iloc[test_idx].copy()
            test_subject = test_df["subject_id"].iloc[0]
            print(f"Held-out Test Subject: {test_subject} ({len(test_df)} trials)")

            train_subjects = sorted(train_val_df["subject_id"].unique())
            val_rng = random.Random(seed + outer_fold)
            val_subject = val_rng.choice(train_subjects)
            print(f"Selected Inner Validation Subject: {val_subject}")

            train_fold_df = train_val_df[train_val_df["subject_id"] != val_subject]
            val_fold_df = train_val_df[train_val_df["subject_id"] == val_subject]

            print(f"Train samples: {len(train_fold_df)} | Val samples: {len(val_fold_df)} | Test samples: {len(test_df)}")

            train_gen = BCI2aDataGenerator(train_fold_df, m_name, batch_size, shuffle=True, normalization=normalization)
            val_gen = BCI2aDataGenerator(val_fold_df, m_name, batch_size, shuffle=False, normalization=normalization)
            test_gen = BCI2aDataGenerator(test_df, m_name, batch_size, shuffle=False, normalization=normalization)

            if m_name == "eegnet":
                model = build_eegnet()
            elif m_name == "cvt":
                model = build_cvt()
            elif m_name == "vit_pe":
                model = build_vit_pe()

            fold_model_dir = os.path.join(output_dir, "models", m_name, f"fold_{outer_fold + 1:02d}")
            os.makedirs(fold_model_dir, exist_ok=True)
            weights_path = os.path.join(fold_model_dir, "best.weights.h5")

            callbacks = [
                EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=1),
                ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
                ModelCheckpoint(weights_path, monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=0)
            ]

            start_time = time.time()
            model.fit(train_gen, validation_data=val_gen, epochs=epochs, callbacks=callbacks, verbose=1)
            train_duration = time.time() - start_time

            y_true, y_pred, y_prob = [], [], []
            for X_b, y_b in test_gen:
                preds = model.predict(X_b, verbose=0)
                y_true.extend(np.argmax(y_b, axis=1))
                y_pred.extend(np.argmax(preds, axis=1))
                y_prob.extend(preds[:, 1])

            y_true = np.array(y_true)
            y_pred = np.array(y_pred)
            y_prob = np.array(y_prob)

            acc = accuracy_score(y_true, y_pred)
            prec = precision_score(y_true, y_pred, zero_division=0)
            rec = recall_score(y_true, y_pred, zero_division=0)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            
            try:
                auc = roc_auc_score(y_true, y_prob)
            except ValueError:
                auc = float('nan')

            cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

            print(f"Test Subject {test_subject} Results -> Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f} | Spec: {specificity:.4f}")

            fold_results.append({
                "model": m_name,
                "outer_fold": outer_fold + 1,
                "test_subject": int(test_subject),
                "val_subject": int(val_subject),
                "accuracy": float(acc),
                "precision": float(prec),
                "recall": float(rec),
                "specificity": float(specificity),
                "f1": float(f1),
                "roc_auc": float(auc),
                "train_duration_sec": float(train_duration)
            })

            for i, row in test_df.iterrows():
                all_predictions.append({
                    "model": m_name,
                    "outer_fold": outer_fold + 1,
                    "test_subject": int(test_subject),
                    "file": row["file"],
                    "y_true": int(row["label"]),
                    "y_pred": int(y_pred[i - test_df.index[0]]),
                    "prob_class_1": float(y_prob[i - test_df.index[0]])
                })

            del model
            tf.keras.backend.clear_session()
            gc.collect()

        res_df = pd.DataFrame(fold_results)
        res_csv = os.path.join(output_dir, f"{m_name}_loso_fold_performance.csv")
        res_df.to_csv(res_csv, index=False)

        pred_df = pd.DataFrame(all_predictions)
        pred_csv = os.path.join(output_dir, f"{m_name}_loso_predictions.csv")
        pred_df.to_csv(pred_csv, index=False)

        print(f"\n[{m_name.upper()}] LOSO Evaluation Completed.")
        print(f"Mean Accuracy: {res_df['accuracy'].mean() * 100:.2f}% +/- {res_df['accuracy'].std() * 100:.2f}%")
        print(f"Mean F1-Score: {res_df['f1'].mean() * 100:.2f}% +/- {res_df['f1'].std() * 100:.2f}%")

    print("\nAll requested models processed successfully.")


def is_running_in_notebook():
    try:
        shell = get_ipython().__class__.__name__
        return shell in ['ZMQInteractiveShell', 'TerminalInteractiveShell']
    except NameError:
        return False


if __name__ == "__main__":
    if is_running_in_notebook():
        default_metadata = os.path.join("..", "data_bci2a_processed_1000", "dataset_metadata_bci2a.csv")
        default_output = os.path.join("..", "results")
        
        print("Running directly inside Jupyter Notebook environment...")
        run_pipeline(
            metadata_path=default_metadata,
            output_dir=default_output,
            model_name="all",
            epochs=50,
            batch_size=32,
            patience=10,
            seed=42,
            normalization="none"
        )
    else:
        parser = argparse.ArgumentParser(description="BCI-IV-2a Strict LOSO Training Pipeline")
        parser.add_argument("--metadata", required=True, help="Path to metadata CSV (file, label, subject_id)")
        parser.add_argument("--model", default="all", choices=["eegnet", "cvt", "vit_pe", "all"])
        parser.add_argument("--output-dir", required=True, help="Directory to save weights, logs, and CSVs")
        parser.add_argument("--epochs", type=int, default=50)
        parser.add_argument("--batch-size", type=int, default=32)
        parser.add_argument("--patience", type=int, default=10)
        parser.add_argument("--seed", type=int, default=42)
        parser.add_argument("--normalization", default="none", choices=["none", "per_trial_channel_zscore"])
        args = parser.parse_args()

        run_pipeline(
            metadata_path=args.metadata,
            output_dir=args.output_dir,
            model_name=args.model,
            epochs=args.epochs,
            batch_size=args.batch_size,
            patience=args.patience,
            seed=args.seed,
            normalization=args.normalization
        )

Running directly inside Jupyter Notebook environment...
Loading metadata from ../data_bci2a_processed_1000/dataset_metadata_bci2a.csv...
Loaded 2538 trials across 9 subjects.

Starting LOSO Training for Model: EEGNET

--- Outer Fold 1 / 9 ---
Held-out Test Subject: 1 (281 trials)
Selected Inner Validation Subject: 3
Train samples: 1975 | Val samples: 282 | Test samples: 281
Epoch 1/50
62/62 [==============================] - 28s 444ms/step - loss: 0.6992 - accuracy: 0.4967 - val_loss: 0.6931 - val_accuracy: 0.4787 - lr: 1.7500e-04
Epoch 2/50
62/62 [==============================] - 19s 306ms/step - loss: 0.6892 - accuracy: 0.5200 - val_loss: 0.6932 - val_accuracy: 0.5035 - lr: 1.7500e-04
Epoch 3/50
62/62 [==============================] - 19s 306ms/step - loss: 0.6782 - accuracy: 0.5646 - val_loss: 0.6928 - val_accuracy: 0.5071 - lr: 1.7500e-04
Epoch 4/50
62/62 [==============================] - 19s 304ms/step - loss: 0.6699 - accuracy: 0.5787 - val_loss: 0.6907 - val_accuracy: 0.5035 